## Notebook del Processing Job
El notebook debe cubrir:

- Setup: sesión de SageMaker, IAM role, bucket y prefix.
- Carga del dataset a S3: sube tus datos crudos al bucket de SageMaker.
- Ejecución del Processing Job.
- Inspección del output: lee las primeras filas del CSV transformado desde S3 para verificar que el job fue exitoso.


In [1]:
## Setup: SageMaker Session, IAM Role y Bucket S3
from time import gmtime, strftime
import sagemaker

sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
bucket = sagemaker_session.default_bucket()
default_bucket_prefix = sagemaker_session.default_bucket_prefix
timestamp_prefix = strftime("%Y-%m-%d-%H-%M-%S", gmtime())

# Configurar rutas S3 para el Processing Job
prefix = "sagemaker/processing-data"

# If a default bucket prefix is specified, append it to the s3 path
if default_bucket_prefix:
    prefix = f"{default_bucket_prefix}/{prefix}"

# S3 paths para el Processing Job
input_prefix = prefix + "/input"
output_prefix = prefix + "/output"

# Rutas dentro del container del Processing Job
input_container_path = "/opt/ml/processing/input"
output_container_path = "/opt/ml/processing/output"

print(f"Bucket: {bucket}")
print(f"Input S3 path: s3://{bucket}/{input_prefix}")
print(f"Output S3 path: s3://{bucket}/{output_prefix}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
Bucket: sagemaker-us-east-1-956463123241
Input S3 path: s3://sagemaker-us-east-1-956463123241/sagemaker/processing-data/input
Output S3 path: s3://sagemaker-us-east-1-956463123241/sagemaker/processing-data/output


### Descarga del dataset y carga a Amazon Simple Storage Service (Amazon S3)

In [3]:
!ls

note_book.ipynb


In [4]:
import boto3
import pandas as pd

# Ruta local de los datos
local_data_path = "../../data/raw"
s3 = boto3.client("s3")

region = sagemaker_session.boto_region_name
f"s3://{bucket}/{input_prefix}"
#input_data = "s3://sagemaker-sample-data-{}/{input_prefix}".format(region)
input_data = f"s3://{bucket}/{input_prefix}".format(region)
print(f"Usando ruta S3: {input_data}")

# Uploading the training data to S3
s3_data_uri = sagemaker_session.upload_data(
    path=local_data_path,
    bucket=bucket,
    key_prefix=input_prefix)
print(f" Datos cargados a: {s3_data_uri}\n")

!aws s3 cp $input_data .

# Verifiacación de datos subidos
response = sagemaker_session.boto_session.client("s3").list_objects_v2(
        Bucket=bucket, 
        Prefix=input_prefix
    )
print(f"\n Archivos en S3:")
if "Contents" in response:
    for obj in response["Contents"]:
        print(f"  ok {obj['Key']}")
        

Usando ruta S3: s3://sagemaker-us-east-1-956463123241/sagemaker/processing-data/input
 Datos cargados a: s3://sagemaker-us-east-1-956463123241/sagemaker/processing-data/input

fatal error: An error occurred (404) when calling the HeadObject operation: Key "sagemaker/processing-data/input" does not exist

 Archivos en S3:
  ok sagemaker/processing-data/input/item_categories.csv
  ok sagemaker/processing-data/input/item_categories_en.csv
  ok sagemaker/processing-data/input/items.csv
  ok sagemaker/processing-data/input/items_en.csv
  ok sagemaker/processing-data/input/sales_train.csv
  ok sagemaker/processing-data/input/shops.csv
  ok sagemaker/processing-data/input/shops_en.csv
  ok sagemaker/processing-data/input/submission.csv
  ok sagemaker/processing-data/input/test.csv


## Construcción del container {#container}

El container BYOC es una imagen Python slim con scikit-learn, pandas y numpy.
No requiere ningún proceso de bootstrapping — SageMaker inyecta y ejecuta el script
directamente con `python3`.

In [5]:
!pwd

/home/sagemaker-user/Arquitectura/processing/notebooks


In [6]:
%cd ../container
!docker build --network sagemaker -t sagemaker-sklearn-preprocess .
%cd ../

/home/sagemaker-user/Arquitectura/processing/container
DEPRECATED: The legacy builder is deprecated and will be removed in a future release.
            BuildKit is currently disabled; enable it by removing the DOCKER_BUILDKIT=0
            environment-variable.

Sending build context to Docker daemon  3.072kB
Step 1/5 : FROM python:3.11-slim
3.11-slim: Pulling from library/python

56c42440: Pulling fs layer 
09444425: Pulling fs layer 
18262cbe: Pulling fs layer 
Digest: sha256:d6e4d224f70f9e0172a06a3a2eba2f768eb146811a349278b38fff3a36463b47
Status: Downloaded newer image for python:3.11-slim
 ---> 7c68b5683872
Step 2/5 : ENV PYTHONHASHSEED 0
 ---> Running in 6148653236d5
 ---> Removed intermediate container 6148653236d5
 ---> a6dc591cb8da
Step 3/5 : ENV PYTHONIOENCODING UTF-8
 ---> Running in 4440db1ec716
 ---> Removed intermediate container 4440db1ec716
 ---> afda23b8c63c
Step 4/5 : RUN pip install --no-cache-dir     numpy==2.4.2     pandas==3.0.0     scikit-learn==1.8.0
 ---> Runni

In [ ]:
## Ejecutar Processing Job con rutas de SageMaker
from sagemaker.processing import ScriptProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput

# Crear el ScriptProcessor para ejecutar el script de preprocessing
script_processor = ScriptProcessor(
    role=role,
    instance_type="ml.m5.xlarge",
    instance_count=1,
    base_job_name="preprocessing-job",
    image_uri="YOUR_ECR_IMAGE_URI"  # Reemplazar con tu imagen Docker en ECR
)

# Configurar inputs del Processing Job
processing_inputs = [
    ProcessingInput(
        source=s3_data_uri,
        destination=input_container_path,
        s3_data_distribution_type="FullyReplicated"
    )
]

# Configurar outputs del Processing Job
processing_outputs = [
    ProcessingOutput(
        source=output_container_path,
        destination=f"s3://{bucket}/{output_prefix}",
        s3_upload_mode="EndOfJob"
    )
]

# Argumentos para el script de preprocessing
script_args = [
    "--input-path", input_container_path,
    "--output-path", output_container_path
]

# Ejecutar el Processing Job
print("Iniciando Processing Job...")
script_processor.run(
    code="../../container/preprocess.py",  # Ruta al script de preprocessing
    inputs=processing_inputs,
    outputs=processing_outputs,
    arguments=script_args
)

print("Processing Job completado!")

In [ ]:
## Verificar outputs del Processing Job
import pandas as pd

# Crear cliente S3
s3_client = sagemaker_session.boto_session.client("s3")
output_s3_path = f"s3://{bucket}/{output_prefix}"

print(f"Archivos en {output_s3_path}:")
response = s3_client.list_objects_v2(Bucket=bucket, Prefix=output_prefix)

if "Contents" in response:
    for obj in response["Contents"]:
        print(f"  - {obj['Key']}")
        
# Descargar y verificar el archivo procesado
output_file_key = f"{output_prefix}/datos_entreno.parquet"  # Ajustar según el archivo generado
local_output_path = "processed_data.parquet"

try:
    s3_client.download_file(bucket, output_file_key, local_output_path)
    df = pd.read_parquet(local_output_path)
    print(f"\nPrimeras filas del archivo procesado:")
    print(df.head())
    print(f"\nForma del dataset: {df.shape}")
    print(f"Columnas: {df.columns.tolist()}")
except Exception as e:
    print(f"✗ Error: {e}")